In [1]:
import pandas as pd
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import Holiday, AbstractHolidayCalendar

# Oslo Børs spesifikke helligdager (eksempel)
class OsloBorsCalendar(AbstractHolidayCalendar):
    rules = [
        Holiday("New Year's Day", month=1, day=1),
        Holiday("Labour Day", month=5, day=1),
        Holiday("Constitution Day", month=5, day=17),
        Holiday("Christmas Day", month=12, day=25),
        Holiday("Boxing Day", month=12, day=26),
        # Legg til flere helligdager spesifikke for Oslo Børs
    ]

# Custom business day for Oslo Børs
oslo_bors_business_day = CustomBusinessDay(calendar=OsloBorsCalendar())

# Lag en funksjon for å filtrere data basert på handelsdager
def adjust_to_trading_days(df, date_column):
    df[date_column] = pd.to_datetime(df[date_column])
    trading_days = pd.date_range(start=df[date_column].min(),
                                 end=df[date_column].max(),
                                 freq=oslo_bors_business_day)
    return df[df[date_column].isin(trading_days)]

# Les Excel-filen
file_path = '/Users/k.a.h/Desktop/MasterData/Factor_Data /Factors Norway - Spread.xlsx'
data = pd.ExcelFile(file_path)

# Behandle hvert ark
adjusted_sheets = {}
for sheet_name in data.sheet_names:
    df = data.parse(sheet_name)
    if 'Date' in df.columns:  # Sørg for at det finnes en dato-kolonne
        adjusted_df = adjust_to_trading_days(df, 'Date')
        adjusted_sheets[sheet_name] = adjusted_df

# Lagre tilbake til en ny Excel-fil
output_path = '/Users/k.a.h/Desktop/MasterData/Factor_Data /Factors_Norway_Adjusted.xlsx'
with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in adjusted_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Justert datasett er lagret til: {output_path}")


Justert datasett er lagret til: /Users/k.a.h/Desktop/MasterData/Factor_Data /Factors_Norway_Adjusted.xlsx
